# SDR Analytics: US Interest Rate Swaps (USD)

This notebook builds a comprehensive SDR analytics suite for a USD interest rate swaps desk. It uses the existing SDR data fetcher and the rateslib-backed pricing infrastructure in the repo to generate volume, rate, and risk analytics.


In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
from pathlib import Path
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.SDRDataBuilder import SDRDataBuilder
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from Query.IRSwaps.backends.rateslib.rl_curve_definitions_map import RATESLIB_CURVE_DEFINITIONS


## 1. Fetch SDR trades

Adjust the intraday window below as needed. The SDR data builder will cache responses under `.cache/sdr` so subsequent runs are faster.


In [ ]:
cache_path = Path(".cache/sdr").resolve()
cache_path.mkdir(parents=True, exist_ok=True)

sdr = SDRDataBuilder(cache_path=str(cache_path), show_tqdm=True)

trade_date = datetime.date(2025, 12, 19)
start = NY_tz.localize(datetime.datetime(trade_date.year, trade_date.month, trade_date.day, 7, 0))
end = NY_tz.localize(datetime.datetime(trade_date.year, trade_date.month, trade_date.day, 17, 0))

raw_trades = sdr.grab_sdr_trades(
    start_timestamp=start,
    end_timestamp=end,
    agency="CFTC",
    asset_class="RATES",
)
raw_trades.tail(5)


## 2. Normalize and filter to USD SOFR/OIS swaps

We normalize notionals, fixed rates, and dates. The filter below keys off the SDR UPI list used in the existing curve definitions.


In [ ]:
def _to_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype(str).str.replace(',', ''), errors="coerce")

trades = raw_trades.copy()
trades["Notional amount-Leg 1"] = _to_numeric(trades["Notional amount-Leg 1"])
trades["Notional amount-Leg 2"] = _to_numeric(trades["Notional amount-Leg 2"])
trades["Fixed rate-Leg 1"] = _to_numeric(trades["Fixed rate-Leg 1"])
trades["Spread-Leg 2"] = _to_numeric(trades["Spread-Leg 2"])

trades["Effective Date"] = pd.to_datetime(trades["Effective Date"], errors="coerce")
trades["Expiration Date"] = pd.to_datetime(trades["Expiration Date"], errors="coerce")
trades["Execution Timestamp"] = pd.to_datetime(trades["Execution Timestamp"], errors="coerce")

trades["Notional"] = trades[["Notional amount-Leg 1", "Notional amount-Leg 2"]].max(axis=1)
trades["Tenor Years"] = (trades["Expiration Date"] - trades["Effective Date"]).dt.days / 365.25

usd_sofr_upis = RATESLIB_CURVE_DEFINITIONS["USD-SOFR-1D"]["SDR_UPIs"]
filtered = trades[
    (trades["Notional currency-Leg 1"] == "USD")
    & (trades["Asset Class"] == "IR")
    & (trades["Unique Product Identifier"].isin(usd_sofr_upis))
    & trades["Fixed rate-Leg 1"].notna()
    & trades["Effective Date"].notna()
    & trades["Expiration Date"].notna()
].copy()

filtered.head()


## 3. Desk-level KPIs

Aggregate the SDR tape for top-line desk metrics.


In [ ]:
kpis = {
    "Trade Count": len(filtered),
    "Total Notional (USD bn)": filtered["Notional"].sum() / 1e9,
    "Median Notional (USD mm)": filtered["Notional"].median() / 1e6,
    "Average Fixed Rate (bp)": filtered["Fixed rate-Leg 1"].mean() * 1e4,
    "Cleared Share (%)": 100 * (filtered["Cleared"] == "Y").mean(),
    "Block Share (%)": 100 * (filtered["Block trade election indicator"] == "True").mean(),
}
pd.Series(kpis).to_frame("Value")


## 4. Tape activity (time-of-day)

Track trading intensity across the session.


In [ ]:
intraday = filtered.copy()
intraday["Execution Timestamp"] = intraday["Execution Timestamp"].dt.tz_convert(NY_tz)
intraday["Hour"] = intraday["Execution Timestamp"].dt.floor("H")

hourly = (
    intraday.groupby("Hour")
    .agg(trades=("Dissemination Identifier", "count"), notional=("Notional", "sum"))
    .reset_index()
)

fig = px.bar(hourly, x="Hour", y="notional", title="USD SOFR OIS Notional by Hour")
fig.update_yaxes(title="Notional (USD)")
fig.show()


## 5. Tenor distribution and rate levels

Bucket tenors to understand where the tape is concentrated, and compare rates across the curve.


In [ ]:
tenor_bins = [0, 1, 2, 5, 10, 15, 20, 30, 50]
tenor_labels = ["0-1Y", "1-2Y", "2-5Y", "5-10Y", "10-15Y", "15-20Y", "20-30Y", "30Y+"]
filtered["Tenor Bucket"] = pd.cut(filtered["Tenor Years"], bins=tenor_bins, labels=tenor_labels, right=False)

tenor_summary = (
    filtered.groupby("Tenor Bucket", dropna=False)
    .agg(trades=("Dissemination Identifier", "count"), notional=("Notional", "sum"), avg_rate=("Fixed rate-Leg 1", "mean"))
    .reset_index()
)
tenor_summary


In [ ]:
fig = px.bar(tenor_summary, x="Tenor Bucket", y="notional", title="Notional by Tenor Bucket")
fig.update_yaxes(title="Notional (USD)")
fig.show()

fig = px.line(tenor_summary, x="Tenor Bucket", y="avg_rate", title="Average Fixed Rate by Tenor")
fig.update_yaxes(title="Fixed Rate")
fig.show()


## 6. Platform, clearing, and execution style

Monitor where liquidity is traded.


In [ ]:
platform_summary = (
    filtered.groupby("Platform identifier")
    .agg(trades=("Dissemination Identifier", "count"), notional=("Notional", "sum"))
    .sort_values("notional", ascending=False)
    .reset_index()
)
platform_summary.head(10)


In [ ]:
fig = px.bar(platform_summary.head(12), x="Platform identifier", y="notional", title="Top Platforms by Notional")
fig.update_yaxes(title="Notional (USD)")
fig.show()

clearing_summary = (
    filtered.groupby("Cleared")
    .agg(trades=("Dissemination Identifier", "count"), notional=("Notional", "sum"))
    .reset_index()
)
fig = px.pie(clearing_summary, values="notional", names="Cleared", title="Cleared vs Uncleared Notional")
fig.show()


## 7. Pricing analytics using the existing curve infrastructure

We pull a USD SOFR curve from the existing `IRSwapsMDP` and compute par-rate spreads and PV01 to quantify risk.


In [ ]:
mdp = IRSwapsMDP(source="CME_NY_EOD_LIVE-RL_BASIC")
curve = mdp.get_pricer({"curve_name": "USD-SOFR-1D", "timestamp": trade_date})

max_priced_trades = 2000
pricing_sample = filtered.head(max_priced_trades).copy()

def price_trade(row: pd.Series) -> pd.Series:
    irs = curve.build_irswap(
        effective_date=row["Effective Date"].date(),
        maturity_date=row["Expiration Date"].date(),
        fixed_rate=row["Fixed rate-Leg 1"],
        notional=row["Notional"],
    )
    par_rate = curve.fair_rate(irs)
    pv01 = curve.pv01(irs)
    return pd.Series({"Par Rate": par_rate, "PV01": pv01})

pricing_metrics = pricing_sample.apply(price_trade, axis=1)
pricing_sample = pd.concat([pricing_sample.reset_index(drop=True), pricing_metrics], axis=1)
pricing_sample["Rate Spread (bp)"] = (pricing_sample["Fixed rate-Leg 1"] - pricing_sample["Par Rate"]) * 1e4
pricing_sample.head()


In [ ]:
fig = px.scatter(
    pricing_sample,
    x="Tenor Years",
    y="Rate Spread (bp)",
    size="Notional",
    color="Cleared",
    title="Trade Rate vs Par (bp)",
)
fig.update_yaxes(title="Rate Spread (bp)")
fig.update_xaxes(title="Tenor (Years)")
fig.show()


In [ ]:
pv01_by_bucket = (
    pricing_sample.groupby("Tenor Bucket")
    .agg(pv01=("PV01", "sum"), notional=("Notional", "sum"))
    .reset_index()
)
fig = px.bar(pv01_by_bucket, x="Tenor Bucket", y="pv01", title="PV01 by Tenor Bucket (Sample)")
fig.update_yaxes(title="PV01 (USD/bp)")
fig.show()


## 8. SDR microstructure: size vs rate

Identify where large notional trades are clearing relative to the curve.


In [ ]:
pricing_sample["Notional (USD mm)"] = pricing_sample["Notional"] / 1e6
fig = px.scatter(
    pricing_sample,
    x="Notional (USD mm)",
    y="Rate Spread (bp)",
    color="Tenor Bucket",
    title="Rate Spread vs Notional",
)
fig.update_xaxes(type="log", title="Notional (USD mm, log scale)")
fig.update_yaxes(title="Rate Spread (bp)")
fig.show()


## 9. Export analytics tables

Store the core analytics for desk reporting.


In [ ]:
output_dir = Path("analytics_outputs")
output_dir.mkdir(exist_ok=True)

kpi_df = pd.Series(kpis).to_frame("Value")
kpi_df.to_csv(output_dir / "kpis.csv")
tenor_summary.to_csv(output_dir / "tenor_summary.csv", index=False)
platform_summary.to_csv(output_dir / "platform_summary.csv", index=False)
pricing_sample.to_csv(output_dir / "pricing_sample.csv", index=False)

output_dir
